# Qwen3 Embedding Experiment Pipeline (Google Colab)

Chào **My**, đây là notebook hướng dẫn và chạy thực nghiệm Embedding cho mô hình **Qwen/Qwen3-Embedding-0.6B** kết hợp với 4 chiến thuật chunking:
1. `token`: baseline theo token có overlap.
2. `langchain_recursive`: recursive character chunking.
3. `llamaindex`: sentence splitter.
4. `structured`: structure-based theo paragraph/structure.

### Hướng dẫn chạy trên Google Colab:
1. Chọn Runtime là **GPU** (Ví dụ: T4 GPU) để tăng tốc độ embedding.
2. Chạy lần lượt các cell bên dưới.

In [ ]:
# 1. Clone repo từ Github (Lấy cụ thể nhánh Qwen chứa các cập nhật mới nhất)
import os
import sys

if not os.path.exists("Text-Mining---RAG-on-News"):
    # Clone cụ thể nhánh Qwen
    !git clone -b Qwen https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git
    %cd Text-Mining---RAG-on-News
else:
    %cd Text-Mining---RAG-on-News
    !git checkout Qwen
    !git pull

# Thêm thư mục hiện tại vào sys.path để Colab nhận diện được thư mục 'src'
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)
print("Đã cấu hình Python Path:", current_dir)

In [ ]:
# 2. Cài đặt các thư viện cần thiết
# Lưu ý: Qwen3 yêu cầu transformers >= 4.51.0 và sentence-transformers >= 2.7.0. Thêm jedi để giải quyết cảnh báo phụ của ipython.
!pip install -q "transformers>=4.51.0" "sentence-transformers>=2.7.0" accelerate llama-index langchain-text-splitters datasets pandas tqdm tabulate jedi

In [ ]:
# 3. Cấu hình các biến chính
from pathlib import Path

# Cấu hình đường dẫn dữ liệu
RAW_CSV_PATH = Path("Dataset/Create_QA_Vietonline/VietOnlineNews/train_new.csv")
PROCESSED_JSONL = Path("src/embed/output/data/vieonline_news_clean.jsonl")
REVIEW_JSONL = Path("src/embed/output/data/vieonline_news_human_review.jsonl")
CHUNK_OUTPUT_DIR = Path("src/embed/output/chunks")
DENSE_OUTPUT_ROOT = Path("src/embed/output/dense")
SAMPLE_REPORT = Path("src/embed/output/embedding_strategy_samples.md")

# Thử nghiệm trên tập dữ liệu: Đặt None để chạy toàn bộ file train (khuyên dùng khi chạy GPU)
ARTICLE_LIMIT = None 

# Các strategy chunking cần thử nghiệm
STRATEGIES = ["token", "langchain_recursive", "llamaindex", "structured"]

# Cấu hình Embedding
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 32  # Giảm từ 64 xuống 32 để tránh lỗi CUDA OOM trên GPU T4 của Colab
SAMPLE_SIZE = 5

for path in [PROCESSED_JSONL.parent, CHUNK_OUTPUT_DIR, DENSE_OUTPUT_ROOT, SAMPLE_REPORT.parent]:
    path.mkdir(parents=True, exist_ok=True)

print("Cấu hình hoàn tất!")

In [ ]:
# 4. Data Ingestion & Chunking
print("--- Bắt đầu Ingestion ---")
!python -m src.data_ingestion.cli \
    --input {RAW_CSV_PATH} \
    --output {PROCESSED_JSONL} \
    --review-output {REVIEW_JSONL}

print("\n--- Bắt đầu Chunking cho 4 chiến thuật ---")
strategies_str = " ".join(STRATEGIES)
limit_arg = f"--limit {ARTICLE_LIMIT}" if ARTICLE_LIMIT is not None else ""

!python -m src.chunking.cli \
    --input {PROCESSED_JSONL} \
    --output-dir {CHUNK_OUTPUT_DIR} \
    --strategies {strategies_str} \
    --chunk-size 400 \
    --overlap 80 \
    --small-article-chars 1000 \
    {limit_arg}

print("Chunking hoàn tất!")

In [ ]:
# 5. Xem mẫu khác biệt của các strategy trước khi embed (Chạy trực tiếp bằng Python import)
import os
import sys
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.embed.compare_embedding_inputs import build_comparison_report
from IPython.display import Markdown

chunk_files = [CHUNK_OUTPUT_DIR / f"vieonline_news_chunks_{strategy}.jsonl" for strategy in STRATEGIES]
report = build_comparison_report(chunk_files, sample_articles=3)
SAMPLE_REPORT.write_text(report, encoding="utf-8")

Markdown(SAMPLE_REPORT.read_text(encoding="utf-8"))

In [ ]:
# 6. Chạy Embedding cho từng strategy sử dụng Qwen3-Embedding-0.6B (Import trực tiếp)
from src.embed.embed_chunks import embed_chunks_file, detect_prefixes
import torch
import gc

# Trích xuất và hiển thị rõ ràng prompt/instruction của mô hình để ghi chép báo cáo
doc_pref, query_pref = detect_prefixes(MODEL_NAME)
print(f"Bắt đầu chạy embedding bằng model: {MODEL_NAME}")
print("=" * 60)
print("CẤU HÌNH PROMPT / INSTRUCTION CHO MODEL QWEN3:")
print(f"- Instruction/Prompt cho Query (Câu hỏi):\n\"\"\"\n{query_pref}\"\"\"")
print(f"- Instruction/Prompt cho Document (Chunk/Tài liệu):\n\"\"\"\n{doc_pref if doc_pref else '(Trống - Không dùng prefix)'}\"\"\"")
print("=" * 60)

# Tự động ghi nhận thông số này vào file notes.md trong workspace của bạn
notes_content = f"""# Embedding Model Notes - Qwen3\n\n- **Model**: {MODEL_NAME}\n- **Thành viên**: My\n- **Query Prompt/Instruction**:\n```text\n{query_pref}```\n- **Document Prompt/Instruction**: (Trống - không dùng prefix)\n- **Padding side**: left\n- **Max sequence length**: 512\n"""
Path("notes.md").write_text(notes_content, encoding="utf-8")
print("Đã lưu thông số prompt vào file: notes.md")

for strategy, chunk_file in zip(STRATEGIES, chunk_files):
    # Giải phóng bộ nhớ GPU trước mỗi chiến thuật để tránh lỗi CUDA OOM
    gc.collect()
    torch.cuda.empty_cache()
    
    output_dir = DENSE_OUTPUT_ROOT / strategy
    print(f"\n--- Embedding cho strategy: {strategy} ---")
    stats, output_paths, samples = embed_chunks_file(
        input_path=chunk_file,
        output_dir=output_dir,
        model_name=MODEL_NAME,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,  # Chuẩn hóa vector để tính cosine similarity chính xác
        sample_size=SAMPLE_SIZE,
        show_progress=True
    )
print("\nEmbedding toàn bộ các chiến thuật thành công!")

In [ ]:
# 7. Đánh giá chất lượng Retrieval trên 50 QA mẫu từ Dataset/QA_Claude/QA_output.csv
import math
import json
import statistics
import time
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from src.embed.dense_search import load_index, search_dense_index

QA_PATH = Path("Dataset/QA_Claude/QA_output.csv")
QA_SAMPLE_SIZE = 50
EVAL_TOP_K = 10

# Load dữ liệu QA
qa_df = pd.read_csv(QA_PATH)
if "is_possible" in qa_df.columns:
    qa_df = qa_df[qa_df["is_possible"].astype(str).str.lower().isin(["true", "1", "yes"])]
qa_df = qa_df.dropna(subset=["question"]).head(QA_SAMPLE_SIZE).copy()

print("Số lượng câu hỏi mẫu đánh giá:", len(qa_df))

# Helper functions
def parse_qrels(row):
    values = set()
    for col in ["source_article_ids", "article_id"]:
        if col not in row or pd.isna(row[col]):
            continue
        raw = row[col]
        if isinstance(raw, (list, tuple, set)):
            parts = raw
        else:
            text = str(raw).strip()
            try:
                parsed = json.loads(text)
                parts = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                parts = text.replace(";", ",").replace("|", ",").split(",")
        for part in parts:
            value = str(part).strip().strip("[]'\"")
            if value and value.lower() not in {"nan", "none", ""}:
                values.add(value)
    return values

def dcg_at_k(relevances, k):
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances[:k]))

def metrics_for_results(results, relevant_article_ids, k=10):
    retrieved_article_ids = [str(item.get("article_id")) for item in results[:k]]
    relevances = [1 if article_id in relevant_article_ids else 0 for article_id in retrieved_article_ids]
    ideal_relevances = [1] * min(len(relevant_article_ids), k)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    ndcg = dcg_at_k(relevances, k) / ideal_dcg if ideal_dcg else 0.0

    def recall_at(cutoff):
        found = set(retrieved_article_ids[:cutoff]) & relevant_article_ids
        return len(found) / len(relevant_article_ids) if relevant_article_ids else 0.0

    def hit_at(cutoff):
        return 1.0 if set(retrieved_article_ids[:cutoff]) & relevant_article_ids else 0.0

    mrr = 0.0
    for idx, article_id in enumerate(retrieved_article_ids[:k], start=1):
        if article_id in relevant_article_ids:
            mrr = 1.0 / idx
            break

    return {
        "nDCG@10": ndcg,
        "Recall@5": recall_at(5),
        "Recall@10": recall_at(10),
        "MRR@10": mrr,
        "Hit@1": hit_at(1),
        "Hit@5": hit_at(5),
    }

def percentile(values, pct):
    if not values:
        return 0.0
    values = sorted(values)
    index = min(len(values) - 1, math.ceil((pct / 100) * len(values)) - 1)
    return values[index]

def directory_size_mb(path):
    path = Path(path)
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file()) / (1024 * 1024)

# Khởi tạo model SentenceTransformer cho Qwen
# Lưu ý sử dụng padding_side="left" cho Qwen3-Embedding
query_model = SentenceTransformer(
    MODEL_NAME, 
    tokenizer_kwargs={"padding_side": "left"}
)

eval_rows = []
for strategy in STRATEGIES:
    index_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = index_dir / "embedding_stats.json"
    manifest_path = index_dir / "manifest.json"
    if not (index_dir / "embeddings.npy").exists():
        print(f"Skip {strategy}: missing embeddings.npy")
        continue

    # Load index và manifest
    embeddings, metadata = load_index(index_dir)
    with open(stats_path, "r", encoding="utf-8") as f:
        stats = json.load(f)
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
        
    query_prefix = manifest.get("query_prefix")
    query_latencies_ms = []
    metric_rows = []

    for _, qa in qa_df.iterrows():
        relevant_article_ids = parse_qrels(qa)
        if not relevant_article_ids:
            continue
        started = time.perf_counter()
        results = search_dense_index(
            query=str(qa["question"]), 
            embeddings=embeddings, 
            metadata=metadata, 
            encoder=query_model, 
            top_k=EVAL_TOP_K,
            query_prefix=query_prefix
        )
        latency_ms = (time.perf_counter() - started) * 1000
        query_latencies_ms.append(latency_ms)

        metrics = metrics_for_results(results, relevant_article_ids, k=EVAL_TOP_K)
        metric_rows.append(metrics)

    if not metric_rows:
        continue

    aggregate = {key: float(np.mean([row[key] for row in metric_rows])) for key in metric_rows[0]}
    eval_rows.append({
        "Model": MODEL_NAME,
        "Chunking": strategy,
        "nDCG@10": aggregate["nDCG@10"],
        "Recall@10": aggregate["Recall@10"],
        "MRR@10": aggregate["MRR@10"],
        "Latency avg ms": statistics.mean(query_latencies_ms),
        "Index MB": directory_size_mb(index_dir)
    })

eval_df = pd.DataFrame(eval_rows)
if not eval_df.empty:
    eval_df = eval_df.sort_values(
        ["nDCG@10", "Recall@10", "Latency avg ms"],
        ascending=[False, False, True]
    ).reset_index(drop=True)
    eval_df.insert(0, "Rank", range(1, len(eval_df) + 1))

# Xuất bảng Markdown kết quả Leaderboard để copy-paste trực tiếp!
print("\n" + "="*20 + " KẾT QUẢ LEADERBOARD CỦA MY " + "="*20)
print(eval_df.to_markdown(index=False))
print("="*60)